# Benchmarking Core Model Architectures

In real-world workflows, you rarely jump straight to a complex model. You build a standardized benchmarking harness to compare:
1. **Logistic Regression (with L2 Regularization):** The baseline linear model.
2. **Random Forest Classifier:** A bagging ensemble of unconstrained decision trees.
3. **HistGradientBoostingClassifier:** A gradient boosted decision tree (GBDT) ensemble using histogram binning.

### Rules for a Fair Comparison:
- **Same Cross-Validation Folds:** Every model must be evaluated on the exact same splits (`StratifiedKFold`).
- **Pipeline Integrity:** Preprocessing (scaling, encoding) must be wrapped inside each model's pipeline to prevent leakage.
- **Identical Scoring Metric:** Evaluated via ROC-AUC (and PR-AUC for imbalanced checks).
- **Execution Profiling:** We measure both validation score and training runtime.

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

np.random.seed(42)

# 1. Synthesize realistic tabular data (mixed relationships, some noise)
X_raw, y_raw = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.80, 0.20],  # 80/20 class distribution
    random_state=42
)

feature_names = [f"feat_{i}" for i in range(20)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = pd.Series(y_raw, name="target")

# Inject occasional missing values to test pipeline robustness
X.iloc[np.random.choice(len(X), size=100, replace=False), 2] = np.nan
X.iloc[np.random.choice(len(X), size=100, replace=False), 7] = np.nan

# 2. Holdout Test Split (locked for the final audit)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Target distribution in Train:\n{y_train.value_counts(normalize=True).round(3)}")

X_train shape: (4000, 20)
X_test shape:  (1000, 20)
Target distribution in Train:
target
0    0.796
1    0.204
Name: proportion, dtype: float64


---
## Step 1: Define Pipelines for Each Model Family

* **Logistic Regression:** Requires scaling (`StandardScaler`) and imputation because it computes distance-based gradients.
* **Random Forest:** Requires imputation (in standard scikit-learn), but does **not** require feature scaling because tree splits are scale-invariant.
* **HistGradientBoosting:** Handles missing values natively and is scale-invariant.

In [ ]:
# 1. Linear Pipeline (requires imputation + scaling)
pipe_lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 2. Random Forest Pipeline (requires imputation)
pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

# 3. Gradient Boosting Pipeline (native missing value support, no scaling needed)
pipe_gb = Pipeline([
    ('classifier', HistGradientBoostingClassifier(max_iter=100, max_depth=6, random_state=42))
])

models = {
    'Logistic Regression': pipe_lr,
    'Random Forest': pipe_rf,
    'HistGradientBoosting': pipe_gb
}

# why did we not include scalar in the random forest classifier, because its ensemble tree based model right?, same goes for hgbc
# but why not imputer. 

---
## Step 2: The Benchmark Harness (`cross_validate`)

We run all models across the exact same 5-fold `StratifiedKFold` splits. 
We collect:
- **Mean ROC-AUC & Std**
- **Mean PR-AUC (`average_precision`) & Std**
- **Fit Time (in seconds)**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision'
}

benchmark_records = []

for name, pipeline in models.items():
    start_time = time.time()
    
    cv_out = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=False
    )
    
    elapsed_time = time.time() - start_time
    
    benchmark_records.append({
        'Model': name,
        'ROC-AUC Mean': round(cv_out['test_roc_auc'].mean(), 4),
        'ROC-AUC Std': round(cv_out['test_roc_auc'].std(), 4),
        'PR-AUC Mean': round(cv_out['test_pr_auc'].mean(), 4),
        'PR-AUC Std': round(cv_out['test_pr_auc'].std(), 4),
        'Avg Fit Time (s)': round(np.mean(cv_out['fit_time']), 3),
        'Total Wall Time (s)': round(elapsed_time, 3)
    })

df_benchmarks = pd.DataFrame(benchmark_records).sort_values(by='ROC-AUC Mean', ascending=False).reset_index(drop=True)

print("=== BASELINE BENCHMARK LEADERBOARD ===")
display(df_benchmarks)

# ive never seen this thing like mean and std for these curves. and what is that total wall time, and what do we get from this table. 

=== BASELINE BENCHMARK LEADERBOARD ===


,Model,ROC-AUC Mean,ROC-AUC Std,PR-AUC Mean,PR-AUC Std,Avg Fit Time (s),Total Wall Time (s)
0,HistGradientBoosting,0.9718,0.0072,0.9380,0.0187,1.222,10.288
1,Random Forest,0.9626,0.0095,0.9184,0.0247,2.029,11.899
2,Logistic Regression,0.9128,0.0114,0.8249,0.0208,0.016,2.691


---
## Step 3: Auditing the Winner on the Locked Test Set

We take the top-performing model from our validation leaderboard, fit it on the entire `X_train`, and score it once on `X_test`.

In [4]:
# Identify winning pipeline
best_model_name = df_benchmarks.iloc[0]['Model']
best_pipeline = models[best_model_name]

print(f"Selected Best Architecture: {best_model_name}\n")

# Fit on full train data
best_pipeline.fit(X_train, y_train)

# Audit against unseen test set
y_test_probs = best_pipeline.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score, average_precision_score

test_roc = roc_auc_score(y_test, y_test_probs)
test_pr = average_precision_score(y_test, y_test_probs)

print(f"Final Test ROC-AUC: {test_roc:.4f}")
print(f"Final Test PR-AUC:  {test_pr:.4f}")

Selected Best Architecture: HistGradientBoosting

Final Test ROC-AUC: 0.9772
Final Test PR-AUC:  0.9496


---
## Architectural Comparison & When to Use Which

| Model Family | Core Mechanism | Strengths | Weaknesses | Best Use Cases |
| :--- | :--- | :--- | :--- | :--- |
| **Logistic Regression** | Weighted sum of features passed through sigmoid function | Extremely fast, fully explainable coefficients, easy to deploy | Cannot capture non-linear relationships without manual feature engineering | Simple baselines, low-latency APIs, heavily regulated industries (banking credit scoring) |
| **Random Forest** | Bagging (averaging predictions of many parallel, deep trees) | Resilient to overfitting, zero need for feature scaling, handles tabular noise | Slower inference on large ensembles, large model file size | Moderate tabular data, situations where tuning time is limited |
| **Gradient Boosting (GBDT)** | Boosting (building sequential trees that correct prior tree residual errors) | Top-tier tabular accuracy, native NaN handling, small memory footprint | Prone to overfitting if learning rate / depth are not tuned | Competitions, high-stakes tabular predictions (fraud, conversion, churn) |